# HR Employee Churn Analysis

This notebook follows a simple flow: inspect the data, clean it, define hypotheses, test the hypotheses, and then review plots one by one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, spearmanr, ttest_ind

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')

## 1. Load Data

Start by checking the structure of the dataset.

In [ ]:
df = pd.read_csv('hr_attribution.csv')

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print('Column names:')
print(df.columns.tolist())
print('
Data types:')
print(df.dtypes)
print('
First 5 rows:')
print(df.head())
print('
Missing values:')
missing_values = df.isna().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))
print('
Attrition distribution:')
print(df['attrition'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 2. Clean Data

Remove duplicate employee rows and standardize text columns.

In [ ]:
clean_df = df.drop_duplicates(subset=['employee_id']).copy()

for column in clean_df.select_dtypes(include='object').columns:
    clean_df[column] = clean_df[column].astype('string').str.strip()

numeric_columns = ['monthly_income', 'percent_salary_hike', 'bonus_eligible', 'overtime_paid_eligible', 'job_satisfaction', 'environment_satisfaction', 'relationship_satisfaction', 'overtime_hours_per_week', 'performance_rating', 'training_times_last_year', 'promotions_last_5years', 'years_since_last_promotion', 'skills_growth_opportunities', 'manager_rating', 'work_life_balance', 'years_at_company', 'years_in_current_role', 'total_working_years', 'distance_from_home_km', 'remote_work_days_per_week']
for column in numeric_columns:
    if column in clean_df.columns:
        clean_df[column] = pd.to_numeric(clean_df[column], errors='coerce')

print('Rows after removing duplicates:', len(clean_df))
print('Remaining missing values:')
remaining_missing = clean_df.isna().sum()
print(remaining_missing[remaining_missing > 0].sort_values(ascending=False))

## 3. Hypotheses

Working hypotheses for the analysis:

- Job satisfaction is related to churn.
  - H0: Job satisfaction and attrition are independent.
  - H1: Lower job satisfaction is associated with attrition.
- Frequent travel is related to churn.
  - H0: Business travel and attrition are independent.
  - H1: Frequent travel is associated with attrition.
- Overtime hours are related to churn.
  - H0: Overtime hours and attrition are not different across groups.
  - H1: Higher overtime is associated with attrition.
- Years since last promotion is related to churn.
  - H0: Years since last promotion and attrition are not different across groups.
  - H1: A larger promotion gap is associated with attrition.
- Performance rating is related to churn.
  - H0: Performance rating and attrition are not different across groups.
  - H1: Performance rating is associated with attrition.

## 4. Null Handling for Testing Features

Check whether the candidate features show a useful relationship. If not, use mode imputation and stop there.

In [ ]:
analysis_df = clean_df.copy()
analysis_df['salary_hike_bucket'] = pd.cut(analysis_df['percent_salary_hike'], bins=[-np.inf, 5, 10, 15, np.inf], labels=['Low', 'Moderate', 'High', 'Very_High'])
analysis_df['training_bucket'] = pd.cut(analysis_df['training_times_last_year'], bins=[-np.inf, 1, 3, 5, np.inf], labels=['Low', 'Moderate', 'High', 'Very_High'])
analysis_df['promotion_gap_bucket'] = pd.cut(analysis_df['years_since_last_promotion'], bins=[-np.inf, 1, 3, 5, np.inf], labels=['Recent', 'Moderate', 'Long', 'Very_Long'])
analysis_df['overtime_bucket'] = pd.cut(analysis_df['overtime_hours_per_week'], bins=[-np.inf, 35, 45, 55, np.inf], labels=['Low', 'Moderate', 'High', 'Very_High'])
analysis_df['support_score'] = analysis_df[['work_life_balance', 'environment_satisfaction', 'skills_growth_opportunities']].mean(axis=1)
analysis_df['support_bucket'] = pd.cut(analysis_df['support_score'], bins=[-np.inf, 2, 3, 4, np.inf], labels=['Low', 'Moderate', 'High', 'Very_High'])

print('Feature columns for null-handling checks:')
print(analysis_df[['salary_hike_bucket', 'training_bucket', 'promotion_gap_bucket', 'overtime_bucket', 'support_score', 'support_bucket']].head())

In [ ]:
performance_summary = []
performance_columns = ['training_times_last_year', 'promotions_last_5years', 'years_since_last_promotion', 'bonus_eligible', 'percent_salary_hike', 'manager_rating']
for column in performance_columns:
    pair = analysis_df[['performance_rating', column]].dropna()
    if len(pair) < 3 or pair['performance_rating'].nunique() < 2 or pair[column].nunique() < 2:
        continue
    rho, p_value = spearmanr(pair['performance_rating'], pair[column])
    performance_summary.append((column, round(float(rho), 4), float(p_value)))

performance_summary = pd.DataFrame(performance_summary, columns=['candidate', 'spearman_r', 'p_value'])
print('Performance rating association check:')
print(performance_summary.sort_values('spearman_r', ascending=False))

if performance_summary.empty or performance_summary['spearman_r'].abs().max() < 0.15:
    analysis_df['performance_rating'] = analysis_df['performance_rating'].fillna(analysis_df['performance_rating'].mode(dropna=True).iloc[0])
    print('Performance rating null handling: mode imputation used because the association is weak.')
else:
    analysis_df['performance_rating'] = analysis_df['performance_rating'].fillna(analysis_df.groupby(['training_bucket', 'promotion_gap_bucket', 'salary_hike_bucket'])['performance_rating'].transform('median'))
    if analysis_df['performance_rating'].isna().any():
        analysis_df['performance_rating'] = analysis_df['performance_rating'].fillna(analysis_df['performance_rating'].mode(dropna=True).iloc[0])
    print('Performance rating null handling: grouped median used first, then mode fallback.')

print('Remaining performance_rating missing values:', analysis_df['performance_rating'].isna().sum())

In [ ]:
satisfaction_summary = []
satisfaction_columns = ['percent_salary_hike', 'bonus_eligible', 'overtime_paid_eligible', 'work_life_balance', 'environment_satisfaction', 'overtime_hours_per_week', 'skills_growth_opportunities']
for column in satisfaction_columns:
    pair = analysis_df[['job_satisfaction', column]].dropna()
    if len(pair) < 3 or pair['job_satisfaction'].nunique() < 2 or pair[column].nunique() < 2:
        continue
    rho, p_value = spearmanr(pair['job_satisfaction'], pair[column])
    satisfaction_summary.append((column, round(float(rho), 4), float(p_value)))

satisfaction_summary = pd.DataFrame(satisfaction_summary, columns=['candidate', 'spearman_r', 'p_value'])
print('Job satisfaction association check:')
print(satisfaction_summary.sort_values('spearman_r', ascending=False))

if satisfaction_summary.empty or satisfaction_summary['spearman_r'].abs().max() < 0.15:
    analysis_df['job_satisfaction'] = analysis_df['job_satisfaction'].fillna(analysis_df['job_satisfaction'].mode(dropna=True).iloc[0])
    print('Job satisfaction null handling: mode imputation used because the association is weak.')
else:
    analysis_df['job_satisfaction'] = analysis_df['job_satisfaction'].fillna(analysis_df.groupby(['salary_hike_bucket', 'overtime_bucket', 'support_bucket'])['job_satisfaction'].transform('median'))
    if analysis_df['job_satisfaction'].isna().any():
        analysis_df['job_satisfaction'] = analysis_df['job_satisfaction'].fillna(analysis_df['job_satisfaction'].mode(dropna=True).iloc[0])
    print('Job satisfaction null handling: grouped median used first, then mode fallback.')

print('Remaining job_satisfaction missing values:', analysis_df['job_satisfaction'].isna().sum())
print('Remaining missing values after targeted handling:')
remaining_after_targeted = analysis_df.isna().sum()
print(remaining_after_targeted[remaining_after_targeted > 0].sort_values(ascending=False))

## 5. Hypothesis Testing

Now test the hypotheses using the cleaned analysis frame.

In [ ]:
alpha = 0.05

travel_table = pd.crosstab(analysis_df['business_travel'], analysis_df['attrition'])
chi2_stat, travel_p_value, travel_dof, travel_expected = chi2_contingency(travel_table)
print('Business travel vs attrition')
print(travel_table)
print('chi-square statistic:', round(float(chi2_stat), 4))
print('p-value:', travel_p_value)
print('Decision:', 'Reject H0' if travel_p_value < alpha else 'Fail to reject H0')

In [ ]:
for column, label in [('job_satisfaction', 'Job satisfaction vs attrition'), ('overtime_hours_per_week', 'Overtime hours vs attrition'), ('years_since_last_promotion', 'Years since last promotion vs attrition'), ('performance_rating', 'Performance rating vs attrition')]:
    yes_values = pd.to_numeric(analysis_df.loc[analysis_df['attrition'] == 'Yes', column], errors='coerce').dropna()
    no_values = pd.to_numeric(analysis_df.loc[analysis_df['attrition'] == 'No', column], errors='coerce').dropna()
    stat, p_value = ttest_ind(yes_values, no_values, equal_var=False)
    print(label)
    print('mean for Yes:', round(float(yes_values.mean()), 2))
    print('mean for No:', round(float(no_values.mean()), 2))
    print('t-statistic:', round(float(stat), 4))
    print('p-value:', p_value)
    print('Decision:', 'Reject H0' if p_value < alpha else 'Fail to reject H0')
    print('')

## 6. Exploratory Data Analysis

Each plot is separate and followed by a short written insight.

In [ ]:
plt.figure(figsize=(8, 5))
attrition_counts = analysis_df['attrition'].value_counts().reindex(['No', 'Yes']).fillna(0)
plt.bar(attrition_counts.index, attrition_counts.values, color=['#4C78A8', '#F58518'])
plt.title('Attrition Class Balance')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Insight

The class balance shows whether churn is imbalanced. That matters because a model can otherwise lean too much toward the majority class.

In [ ]:
plt.figure(figsize=(9, 5))
travel_counts = pd.crosstab(analysis_df['business_travel'], analysis_df['attrition']).reindex(['Non-Travel', 'Travel_Rarely', 'Travel_Frequently'])
travel_counts.plot(kind='bar', color=['#4C78A8', '#F58518'], width=0.8)
plt.title('Business Travel by Attrition')
plt.xlabel('Business Travel')
plt.ylabel('Count')
plt.legend(title='Attrition')
plt.tight_layout()
plt.show()

### Insight

Frequent travel is treated as a churn risk signal here, so this plot checks whether the left group is concentrated among frequent travelers.

In [ ]:
plt.figure(figsize=(8, 5))
job_sat_no = analysis_df.loc[analysis_df['attrition'] == 'No', 'job_satisfaction'].dropna()
job_sat_yes = analysis_df.loc[analysis_df['attrition'] == 'Yes', 'job_satisfaction'].dropna()
plt.boxplot([job_sat_no, job_sat_yes], labels=['No', 'Yes'], patch_artist=True, boxprops=dict(facecolor='#A0CBE8', color='#2F4B7C'), medianprops=dict(color='#E45756', linewidth=2))
plt.title('Job Satisfaction vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Job Satisfaction')
plt.tight_layout()
plt.show()

### Insight

Lower job satisfaction should align with churn if the hypothesis is valid.

In [ ]:
plt.figure(figsize=(8, 5))
overtime_no = analysis_df.loc[analysis_df['attrition'] == 'No', 'overtime_hours_per_week'].dropna()
overtime_yes = analysis_df.loc[analysis_df['attrition'] == 'Yes', 'overtime_hours_per_week'].dropna()
plt.boxplot([overtime_no, overtime_yes], labels=['No', 'Yes'], patch_artist=True, boxprops=dict(facecolor='#54A24B', color='#2F4B7C'), medianprops=dict(color='#E45756', linewidth=2))
plt.title('Overtime Hours vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Overtime Hours per Week')
plt.tight_layout()
plt.show()

### Insight

If overtime is a churn driver, the attrition group should sit at a higher center and show more spread.

In [ ]:
plt.figure(figsize=(8, 5))
promo_no = analysis_df.loc[analysis_df['attrition'] == 'No', 'years_since_last_promotion'].dropna()
promo_yes = analysis_df.loc[analysis_df['attrition'] == 'Yes', 'years_since_last_promotion'].dropna()
plt.boxplot([promo_no, promo_yes], labels=['No', 'Yes'], patch_artist=True, boxprops=dict(facecolor='#F2CF5B', color='#2F4B7C'), medianprops=dict(color='#E45756', linewidth=2))
plt.title('Years Since Last Promotion vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Years Since Last Promotion')
plt.tight_layout()
plt.show()

### Insight

A larger promotion gap should correspond to higher churn risk because long delays in promotion often reduce retention.

In [ ]:
plt.figure(figsize=(8, 5))
performance_no = analysis_df.loc[analysis_df['attrition'] == 'No', 'performance_rating'].dropna()
performance_yes = analysis_df.loc[analysis_df['attrition'] == 'Yes', 'performance_rating'].dropna()
plt.boxplot([performance_no, performance_yes], labels=['No', 'Yes'], patch_artist=True, boxprops=dict(facecolor='#B279A2', color='#2F4B7C'), medianprops=dict(color='#E45756', linewidth=2))
plt.title('Performance Rating vs Attrition')
plt.xlabel('Attrition')
plt.ylabel('Performance Rating')
plt.tight_layout()
plt.show()

### Insight

This plot helps confirm whether performance rating separates churn from non-churn strongly enough to justify more than mode fallback.

## 7. Summary

The notebook now stays simple: short code cells, no helper functions, plain print output, and a separate markdown cell after each plot.